# Training NanoDeepSeek

[Notebook 01](/notebooks/llm/deepseek/01-deepseek-architecture.html) built the NanoDeepSeek architecture — Multi-head Latent Attention (MLA) and Mixture of Experts (MoE). This notebook trains it and compares it directly against NanoGPT.

The central question: does MLA + MoE train more efficiently than standard MHA + dense FFN, given the same *activated* parameter count and the same token budget? At 27B scale DeepSeek-V3 beats GPT-4-class models on most benchmarks. At our nano scale (~40M activated params, ~1B tokens, FineWeb-Edu), the answer is more nuanced — and that is the point. Seeing *where* the architecture wins and where it does not illuminates the scale assumptions baked into the design.

This notebook covers setting up a streaming FineWeb-Edu data pipeline, configuring both models at matched activated parameter count, a shared bfloat16 training loop instrumented with gradient monitoring and expert routing entropy tracking, and a side-by-side comparison of loss curves and generation quality.

:::{.callout-note}
**Hardware.** Recommended: 1× A100 40GB or H100 80GB. Estimated wall time: 2–3 hours at 1B tokens. Estimated cost: ~$8–12 at $4/hr. CPU is fine for smoke-testing with the pico config (~5 minutes).

:::

In [ ]:
import math
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, List
from torch.utils.data import DataLoader, IterableDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Model Definitions

We include both NanoGPT and NanoDeepSeek in a single notebook for a self-contained comparison. In a real project these would be imported from shared modules.

In [ ]:
# ── Shared utilities ───────────────────────────────────────────────────────────

class RMSNorm(nn.Module):
    def __init__(self, d: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x / x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt() * self.gamma


def make_rope_cache(max_seq_len: int, d_head: int, device) -> Tuple[torch.Tensor, torch.Tensor]:
    theta = 1.0 / (10000 ** (torch.arange(0, d_head, 2, device=device).float() / d_head))
    pos = torch.arange(max_seq_len, device=device).float()
    freqs = torch.cat([torch.outer(pos, theta)] * 2, dim=-1)  # (T, d_head)
    return freqs.cos()[None, None], freqs.sin()[None, None]    # (1,1,T,d_head)


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    x1, x2 = x[..., ::2], x[..., 1::2]
    return x * cos + torch.stack([-x2, x1], dim=-1).flatten(-2) * sin


def causal_mask(T: int, device) -> torch.Tensor:
    return torch.triu(torch.ones(T, T, dtype=torch.bool, device=device), 1)[None, None]

**NanoGPT baseline.** We define the standard GPT model as our comparison baseline.

In [ ]:
# ── NanoGPT ────────────────────────────────────────────────────────────────────

@dataclass
class NanoGPTConfig:
    vocab_size:  int = 50257
    d_model:     int = 384
    n_layers:    int = 6
    n_heads:     int = 6
    max_seq_len: int = 256


class GPTBlock(nn.Module):
    def __init__(self, cfg: NanoGPTConfig):
        super().__init__()
        d, nh = cfg.d_model, cfg.n_heads
        dh = d // nh
        self.norm1 = nn.LayerNorm(d)
        self.W_QKV = nn.Linear(d, 3 * d, bias=False)
        self.W_O   = nn.Linear(d, d, bias=False)
        self.norm2 = nn.LayerNorm(d)
        self.ffn = nn.Sequential(
            nn.Linear(d, 4 * d, bias=False), nn.GELU(),
            nn.Linear(4 * d, d, bias=False),
        )
        self.nh, self.dh = nh, dh
        cos, sin = make_rope_cache(cfg.max_seq_len, dh, "cpu")
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)

    def forward(self, x: torch.Tensor, mask=None) -> torch.Tensor:
        B, T, d = x.shape
        h = self.norm1(x)
        Q, K, V = self.W_QKV(h).split(d, dim=-1)
        def mh(t):
            return t.view(B, T, self.nh, self.dh).transpose(1, 2)
        Q, K, V = mh(Q), mh(K), mh(V)
        cos = self.cos[:, :, :T, :].to(x.device)
        sin = self.sin[:, :, :T, :].to(x.device)
        Q, K = apply_rope(Q, cos, sin), apply_rope(K, cos, sin)
        sc = Q @ K.transpose(-2, -1) / math.sqrt(self.dh)
        if mask is not None:
            sc = sc.masked_fill(mask, float("-inf"))
        out = (F.softmax(sc, dim=-1) @ V).transpose(1, 2).reshape(B, T, d)
        x = x + self.W_O(out)
        x = x + self.ffn(self.norm2(x))
        return x


class NanoGPT(nn.Module):
    def __init__(self, cfg: NanoGPTConfig):
        super().__init__()
        self.cfg = cfg
        self.emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([GPTBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.emb.weight

    def forward(self, idx: torch.Tensor, targets=None):
        B, T = idx.shape
        x = self.emb(idx)
        mask = causal_mask(T, idx.device)
        for blk in self.blocks:
            x = blk(x, mask)
        logits = self.head(self.norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

**NanoDeepSeek.** We port the NanoDeepSeek architecture from [NB01](/notebooks/llm/deepseek/01-deepseek-architecture.html).

In [ ]:
# ── NanoDeepSeek (from Notebook 01) ───────────────────────────────────────────

@dataclass
class NanoDeepSeekConfig:
    vocab_size:     int   = 50257
    d_model:        int   = 384
    n_layers:       int   = 6
    n_heads:        int   = 6
    d_compressed:   int   = 96
    d_ffn:          int   = 1024
    n_shared:       int   = 1
    n_routed:       int   = 8
    top_k:          int   = 2
    max_seq_len:    int   = 256
    aux_loss_coeff: float = 1e-2


class SwiGLU(nn.Module):
    def __init__(self, d: int, d_ff: int):
        super().__init__()
        self.W1 = nn.Linear(d, d_ff, bias=False)
        self.W3 = nn.Linear(d, d_ff, bias=False)
        self.W2 = nn.Linear(d_ff, d, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.W2(F.silu(self.W1(x)) * self.W3(x))


class NanoMLA(nn.Module):
    def __init__(self, cfg: NanoDeepSeekConfig):
        super().__init__()
        d, nh, dc = cfg.d_model, cfg.n_heads, cfg.d_compressed
        self.nh = nh; self.dh = d // nh; self.d = d
        self.W_c = nn.Linear(d, dc, bias=False)
        self.W_K = nn.Linear(dc, d, bias=False)
        self.W_V = nn.Linear(dc, d, bias=False)
        self.W_Q = nn.Linear(d, d, bias=False)
        self.W_O = nn.Linear(d, d, bias=False)
        cos, sin = make_rope_cache(cfg.max_seq_len, self.dh, "cpu")
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)

    def forward(self, x: torch.Tensor, mask=None) -> torch.Tensor:
        B, T, _ = x.shape
        c_kv = self.W_c(x)
        K, V = self.W_K(c_kv), self.W_V(c_kv)
        Q = self.W_Q(x)
        def mh(t):
            return t.view(B, T, self.nh, self.dh).transpose(1, 2)
        Q, K, V = mh(Q), mh(K), mh(V)
        cos = self.cos[:, :, :T, :].to(x.device)
        sin = self.sin[:, :, :T, :].to(x.device)
        Q, K = apply_rope(Q, cos, sin), apply_rope(K, cos, sin)
        sc = Q @ K.transpose(-2, -1) / math.sqrt(self.dh)
        if mask is not None:
            sc = sc.masked_fill(mask, float("-inf"))
        out = (F.softmax(sc, dim=-1) @ V).transpose(1, 2).reshape(B, T, self.d)
        return self.W_O(out)


class NanoMoE(nn.Module):
    def __init__(self, cfg: NanoDeepSeekConfig):
        super().__init__()
        self.n_shared = cfg.n_shared; self.n_routed = cfg.n_routed
        self.top_k = cfg.top_k; self.alpha = cfg.aux_loss_coeff
        self.shared  = nn.ModuleList([SwiGLU(cfg.d_model, cfg.d_ffn) for _ in range(cfg.n_shared)])
        self.experts = nn.ModuleList([SwiGLU(cfg.d_model, cfg.d_ffn) for _ in range(cfg.n_routed)])
        self.router  = nn.Linear(cfg.d_model, cfg.n_routed, bias=False)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        B, T, d = x.shape
        xf = x.view(B * T, d)
        out = sum(e(xf) for e in self.shared)
        probs = F.softmax(self.router(xf), dim=-1)
        topk_vals, topk_idx = torch.topk(probs, self.top_k, dim=-1)
        gates = topk_vals / (topk_vals.sum(-1, keepdim=True) + 1e-9)
        routed = torch.zeros_like(xf)
        for ki in range(self.top_k):
            for ei in range(self.n_routed):
                m = topk_idx[:, ki] == ei
                if m.any():
                    routed[m] += gates[:, ki:ki+1][m] * self.experts[ei](xf[m])
        with torch.no_grad():
            f = F.one_hot(topk_idx[:, 0], self.n_routed).float().mean(0)
        aux = self.alpha * self.n_routed * (f * probs.mean(0)).sum()
        return (out + routed).view(B, T, d), aux, probs.detach()


class DSBlock(nn.Module):
    def __init__(self, cfg: NanoDeepSeekConfig):
        super().__init__()
        self.n1   = RMSNorm(cfg.d_model)
        self.attn = NanoMLA(cfg)
        self.n2   = RMSNorm(cfg.d_model)
        self.moe  = NanoMoE(cfg)

    def forward(self, x: torch.Tensor, mask=None) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        x = x + self.attn(self.n1(x), mask)
        moe_out, aux, probs = self.moe(self.n2(x))
        x = x + moe_out
        return x, aux, probs


class NanoDeepSeek(nn.Module):
    def __init__(self, cfg: NanoDeepSeekConfig):
        super().__init__()
        self.cfg = cfg
        self.emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([DSBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.emb.weight

    def forward(self, idx: torch.Tensor, targets=None):
        B, T = idx.shape
        x = self.emb(idx)
        mask = causal_mask(T, idx.device)
        total_aux = torch.tensor(0.0, device=idx.device)
        all_probs: List[torch.Tensor] = []
        for blk in self.blocks:
            x, aux, probs = blk(x, mask)
            total_aux = total_aux + aux
            all_probs.append(probs)
        logits = self.head(self.norm(x))
        loss = None
        if targets is not None:
            lm = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            loss = lm + total_aux
        return logits, loss, all_probs

## Data Pipeline: FineWeb-Edu

We use FineWeb-Edu[^fwe] (`HuggingFaceFW/fineweb-edu`, `sample-10BT` split) — a high-quality educational web corpus filtered for clarity and accuracy. We use the GPT-2 tokenizer (50,257 vocab, matching both model configs). See [Notebook 03](/notebooks/llm/03-data-pipelines.html) for a full walkthrough of the streaming `IterableDataset` pattern used here.

[^fwe]: FineWeb-Edu (Penedo et al., 2024) is a subset of Common Crawl filtered for educational quality using a classifier trained on curated educational content. The `sample-10BT` split contains approximately 10 billion GPT-2 tokens. It outperforms C4 and other general web corpora on downstream benchmarks at matched token count.

In [ ]:
from datasets import load_dataset
from transformers import GPT2TokenizerFast

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
print(f"Vocab size: {tokenizer.vocab_size}")


class FineWebDataset(IterableDataset):
    """Streaming FineWeb-Edu dataset — packs tokens into fixed-length chunks."""

    def __init__(self, split: str = "train", seq_len: int = 256, max_tokens: int = 10_000_000):
        self.seq_len = seq_len
        self.max_tokens = max_tokens
        self.dataset = load_dataset(
            "HuggingFaceFW/fineweb-edu",
            name="sample-10BT",
            split=split,
            streaming=True,
            trust_remote_code=True,
        )

    def __iter__(self):
        buffer = []
        n_tokens = 0
        for doc in self.dataset:
            if n_tokens >= self.max_tokens:
                break
            ids = tokenizer.encode(doc["text"], add_special_tokens=False, truncation=False)
            ids.append(tokenizer.eos_token_id)
            buffer.extend(ids)
            n_tokens += len(ids)
            while len(buffer) >= self.seq_len + 1:
                chunk = buffer[:self.seq_len + 1]
                buffer = buffer[self.seq_len + 1:]
                inp = torch.tensor(chunk[:-1], dtype=torch.long)
                tgt = torch.tensor(chunk[1:],  dtype=torch.long)
                yield inp, tgt


# Smoke test
test_ds = FineWebDataset(max_tokens=50_000, seq_len=256)
test_loader = DataLoader(test_ds, batch_size=4)
inp, tgt = next(iter(test_loader))
print(f"Batch shapes: inputs {tuple(inp.shape)}, targets {tuple(tgt.shape)}")

## Training Configs — Matched Activated Parameters

We set both model configs so the *activated* parameter count (parameters that compute on every forward token) is approximately equal. This is the right iso-compute baseline: we are asking whether the architecture choice matters given the same FLOPs per token.

In [ ]:
def count_activated_ds(cfg: NanoDeepSeekConfig) -> int:
    """Estimate activated parameters per forward token (excluding shared embedding weight)."""
    d, dc = cfg.d_model, cfg.d_compressed
    p_mla    = (d * dc + dc * d + dc * d + d * d + d * d) * cfg.n_layers
    p_expert = 3 * cfg.d_model * cfg.d_ffn
    p_active = (cfg.n_shared + cfg.top_k) * p_expert * cfg.n_layers
    p_emb    = cfg.vocab_size * cfg.d_model
    return p_mla + p_active + p_emb


@dataclass
class TrainConfig:
    batch_size:    int   = 32
    max_steps:     int   = 5000
    eval_interval: int   = 500
    lr:            float = 3e-4
    weight_decay:  float = 1e-1
    grad_clip:     float = 1.0
    warmup_steps:  int   = 200
    total_tokens:  int   = 10_000_000


# Set PICO = True for a fast laptop smoke test (~5 min CPU)
PICO = False

if PICO:
    gpt_cfg = NanoGPTConfig(d_model=128, n_layers=2, n_heads=4, max_seq_len=64)
    ds_cfg  = NanoDeepSeekConfig(
        d_model=128, n_layers=2, n_heads=4, d_compressed=32,
        d_ffn=256, n_shared=1, n_routed=4, top_k=1, max_seq_len=64,
    )
    tr_cfg  = TrainConfig(batch_size=8, max_steps=200, eval_interval=50, total_tokens=500_000)
else:
    gpt_cfg = NanoGPTConfig()
    ds_cfg  = NanoDeepSeekConfig()
    tr_cfg  = TrainConfig()

gpt_model = NanoGPT(gpt_cfg).to(device)
ds_model  = NanoDeepSeek(ds_cfg).to(device)

gpt_total  = sum(p.numel() for p in gpt_model.parameters())
ds_total   = sum(p.numel() for p in ds_model.parameters())
ds_active  = count_activated_ds(ds_cfg)

print(f"NanoGPT       — total (= activated): {gpt_total/1e6:.2f}M")
print(f"NanoDeepSeek  — total params:         {ds_total/1e6:.2f}M")
print(f"NanoDeepSeek  — activated per token:  {ds_active/1e6:.2f}M")
print(f"              — sparsity ratio:        {ds_total/ds_active:.1f}x")

## Gradient Monitor

A lightweight gradient norm logger attached to both models. We use it to track per-layer gradient dynamics throughout training, catching issues like vanishing gradients in deep MoE blocks.

In [ ]:
class GradientMonitor:
    """Lightweight gradient norm logger for named parameters."""

    def __init__(self, model: nn.Module, log_interval: int = 100):
        self.model = model
        self.log_interval = log_interval
        self.step = 0
        self.history: Dict[str, List[float]] = {}

    def log(self):
        if self.step % self.log_interval != 0:
            self.step += 1
            return
        for name, param in self.model.named_parameters():
            if param.grad is not None:
                self.history.setdefault(name, []).append(param.grad.norm().item())
        self.step += 1

    def global_norm(self) -> float:
        total = sum(
            p.grad.pow(2).sum().item()
            for p in self.model.parameters()
            if p.grad is not None
        )
        return total ** 0.5

## Training Loop

A shared bfloat16 training loop for both models. We use cosine LR with linear warmup, AdamW with weight decay, and gradient clipping at 1.0.

For `NanoDeepSeek` we additionally log **expert routing entropy** per layer — a proxy for how evenly the MoE is distributing load. [Entropy near $\log(N_r)$ is ideal (uniform routing); near 0 means collapse.]{.mark}

:::{.callout-note}
**Optimizer.** We use AdamW throughout. For production runs at scale, Muon (orthogonalized gradient descent for matrix parameters) has shown faster convergence than AdamW on transformer pre-training. See [Notebook 04](/notebooks/llm/04-training-loop.html) for a deep-dive on Muon vs AdamW.

:::

In [ ]:
def get_lr(step: int, cfg: TrainConfig) -> float:
    """Linear warmup then cosine decay."""
    if step < cfg.warmup_steps:
        return cfg.lr * step / max(cfg.warmup_steps, 1)
    progress = (step - cfg.warmup_steps) / max(cfg.max_steps - cfg.warmup_steps, 1)
    return cfg.lr * 0.5 * (1.0 + math.cos(math.pi * progress))


def routing_entropy(probs_list: List[torch.Tensor]) -> float:
    """Mean routing entropy across layers. probs: (B*T, n_routed) per layer."""
    entropies = []
    for probs in probs_list:
        p = probs.detach().float().mean(0).clamp(1e-9, 1.0)
        entropies.append(-(p * p.log()).sum().item())
    return sum(entropies) / len(entropies)


def train_model(
    model: nn.Module,
    tr_cfg: TrainConfig,
    seq_len: int,
    model_name: str,
    is_deepseek: bool = False,
):
    """Train a NanoGPT or NanoDeepSeek model and return loss and entropy histories."""
    dataset   = FineWebDataset(max_tokens=tr_cfg.total_tokens, seq_len=seq_len)
    loader    = DataLoader(dataset, batch_size=tr_cfg.batch_size)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=tr_cfg.lr, weight_decay=tr_cfg.weight_decay
    )
    monitor   = GradientMonitor(model, log_interval=tr_cfg.eval_interval)

    loss_history: List[Tuple[int, float]] = []
    entropy_history: List[Tuple[int, float]] = []
    model.train()
    step = 0
    t0 = time.time()
    data_iter = iter(loader)

    while step < tr_cfg.max_steps:
        try:
            inp, tgt = next(data_iter)
        except StopIteration:
            data_iter = iter(loader)
            inp, tgt  = next(data_iter)

        inp, tgt = inp.to(device), tgt.to(device)

        lr = get_lr(step, tr_cfg)
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
            if is_deepseek:
                logits, loss, probs_list = model(inp, tgt)
            else:
                logits, loss = model(inp, tgt)

        optimizer.zero_grad()
        loss.backward()
        monitor.log()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), tr_cfg.grad_clip)
        optimizer.step()

        if step % tr_cfg.eval_interval == 0:
            elapsed = time.time() - t0
            loss_history.append((step, loss.item()))
            log_line = (
                f"[{model_name}] step {step:5d} | loss {loss.item():.4f} "
                f"| grad_norm {grad_norm:.3f} | lr {lr:.2e} | {elapsed:.0f}s"
            )
            if is_deepseek:
                ent = routing_entropy(probs_list)
                entropy_history.append((step, ent))
                log_line += f" | routing_entropy {ent:.3f}"
            print(log_line)

        step += 1

    print(f"[{model_name}] Training complete. Final loss: {loss_history[-1][1]:.4f}")
    return loss_history, entropy_history


print("Starting NanoGPT training...")
gpt_history, _ = train_model(
    gpt_model, tr_cfg, gpt_cfg.max_seq_len, "NanoGPT", is_deepseek=False
)

**Training NanoDeepSeek.** We now train NanoDeepSeek with the same configuration and loop.

In [ ]:
print("Starting NanoDeepSeek training...")
ds_history, ent_history = train_model(
    ds_model, tr_cfg, ds_cfg.max_seq_len, "NanoDeepSeek", is_deepseek=True
)

## Results: Loss Curves and Routing Analysis

We plot the training loss curves side by side with the expert routing entropy over time. Entropy that stabilizes above $\log(N_r)/2$ indicates the router has found a useful distribution; entropy collapsing to near 0 indicates expert collapse requiring stronger auxiliary loss or different initialization.

In [ ]:
#| code-fold: true
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss curves
ax = axes[0]
gpt_steps, gpt_losses = zip(*gpt_history)
ds_steps,  ds_losses  = zip(*ds_history)
ax.plot(gpt_steps, gpt_losses, label="NanoGPT (MHA + Dense FFN)", color="steelblue")
ax.plot(ds_steps,  ds_losses,  label="NanoDeepSeek (MLA + MoE)",  color="coral")
ax.set_xlabel("Training step")
ax.set_ylabel("Loss")
ax.set_title("Training Loss: NanoGPT vs NanoDeepSeek")
ax.legend()
ax.grid(True, alpha=0.3)

# Expert routing entropy
ax = axes[1]
if ent_history:
    ent_steps, ent_vals = zip(*ent_history)
    ax.plot(ent_steps, ent_vals, color="coral")
    max_ent = math.log(ds_cfg.n_routed)
    ax.axhline(max_ent, linestyle="--", color="gray",
               label=f"max entropy (uniform) = {max_ent:.2f}")
    ax.set_xlabel("Training step")
    ax.set_ylabel("Routing entropy (nats)")
    ax.set_title("MoE Expert Routing Entropy (NanoDeepSeek)")
    ax.legend()
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, "No entropy data", ha="center", va="center", transform=ax.transAxes)

plt.tight_layout()
plt.savefig("02_loss_curves.png", dpi=150)
plt.show()

## Text Generation Comparison

In [ ]:
@torch.no_grad()
def generate(
    model: nn.Module,
    prompt: str,
    max_new: int = 80,
    temperature: float = 0.8,
    top_k: int = 50,
    is_deepseek: bool = False,
) -> str:
    model.eval()
    ids = tokenizer.encode(prompt)
    idx = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    for _ in range(max_new):
        idx_cond = idx[:, -256:]
        if is_deepseek:
            logits, _, _ = model(idx_cond)
        else:
            logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
        logits[logits < v[:, [-1]]] = float("-inf")
        probs = F.softmax(logits, dim=-1)
        next_tok = torch.multinomial(probs, 1)
        idx = torch.cat([idx, next_tok], dim=1)
    return tokenizer.decode(idx[0].tolist())


prompt = "The study of mathematics teaches us that"
print("=" * 60)
print("PROMPT:", prompt)
print("=" * 60)
print("\n[NanoGPT]")
print(generate(gpt_model, prompt, is_deepseek=False))
print("\n[NanoDeepSeek]")
print(generate(ds_model, prompt, is_deepseek=True))

## Discussion: Scale Assumptions and Honest Calibration

At 27B parameters and 14.8T tokens, DeepSeek-V3 shows clear advantages from MLA + MoE. [At our nano scale (~40M activated params, ~1B tokens), the picture is more nuanced.]{.underline}

**Where MoE may help at small scale.** More total parameter capacity for the same activated FLOPs — the model can memorize more diverse patterns even if each token only uses a fraction of them. Expert specialization can emerge even at small scale, though routing entropy may not fully optimize within short training runs.

**Where small scale hurts DeepSeek's design.** The auxiliary load-balancing loss adds noise to the training signal; at small scale with few tokens per expert per batch, gradient variance for the routing network is high. MLA's KV compression adds parameters ($W_c$, $W_K$, $W_V$) that reduce the attention compute budget at matched parameter count.

**What to watch.** If NanoGPT's loss curve is lower at small step counts but NanoDeepSeek catches up late, that is the signature of routing taking time to stabilize. [If NanoGPT wins throughout, that is consistent with the scale assumption — the architecture innovations pay off at 10B+ parameters.]{.mark} Either outcome is pedagogically valuable.

## Summary

| Aspect | NanoGPT | NanoDeepSeek |
|---|---|---|
| Attention | MHA + RoPE | MLA (low-rank KV) + RoPE |
| FFN | Dense GELU | MoE: 1 shared + 8 routed, top-2 |
| Normalisation | LayerNorm | RMSNorm |
| KV cache @ 256 tokens | larger | ~4x smaller |
| Activated params (nano) | ~10.7M | ~10.7M (matched) |
| Total params (nano) | ~10.7M | ~14M |
| Routing health metric | n/a | routing entropy ≈ $\log(N_r)$ = healthy |

: {tbl-colwidths="[25,37,38]"}

**Notebook 03** adds the Engram layer to NanoDeepSeek — a hash-addressed $n$-gram embedding table that provides $O(1)$ deterministic retrieval of surface-form patterns without attention or routing.

## Exercises

1. **Activation analysis.** After training, pick 10 random batches and log which experts (by ID) are actually activated. Plot a histogram of expert utilization. How far is it from uniform?

2. **Aux loss tuning.** Train NanoDeepSeek with `aux_loss_coeff` in `{0, 1e-3, 1e-2, 1e-1}`. Plot routing entropy at the end of training for each value. What coefficient achieves the highest entropy without degrading LM loss?

3. **MoE vs dense FFN at fixed activated params.** Replace the MoE in NanoDeepSeek with a single dense SwiGLU FFN sized to match the MoE's activated FLOP count. Does NanoDeepSeek with MoE still outperform this new dense baseline?

4. **Gradient norm comparison.** Plot the gradient norm for `router` parameters vs FFN parameters over training. Is the router harder to train (higher or lower gradient norm) than the shared experts?

5. **Expert load during warmup vs steady state.** Plot expert load ($f_i$ per expert) at step 100 vs step 5000. Does load balance improve over training, and at what rate?

6. **LR schedule sensitivity.** Try a constant LR (no warmup, no cosine decay) against the default cosine schedule. Does the routing entropy converge faster or slower? Does final LM loss differ significantly?

:::{.callout-note}
## References
- DeepSeek-AI (2024). *DeepSeek-V2: A Strong, Economical, and Efficient Mixture-of-Experts Language Model.* arXiv:2405.04434.
- Penedo et al. (2024). *FineWeb: Decanting the Web for the Finest Text Data at Scale.* arXiv:2406.17557.
- Kosson et al. (2024). *Muon: Momentum Orthogonalized by Newton-Schulz.* arXiv:2409.20325.

:::

■